# Are NYC restaurants getting a passing grade?

**Tianju Cao**

New York City publishes restaurant inspection results. An inspection score is a count of violation points: **lower is better**. I wanted to know what a typical score looks like and whether most restaurants cluster at the low end or whether high scores are common.

I use the latest 1,000 inspections with a numeric score from the NYC Department of Health and Mental Hygiene's Restaurant Inspections dataset. This is a sample of recent scored inspections, not a complete census of every restaurant.

## Loading the data

The CSV contains only the columns needed for this analysis: restaurant name, borough, cuisine, inspection date, score, and grade. I downloaded it from the NYC Open Data API and keep it beside the notebook so the analysis can be reproduced.

In [1]:
import pandas as pd

inspections = pd.read_csv('restaurant_inspections.csv')
inspections.head()

,camis,dba,boro,cuisine_description,inspection_date,score,grade
0,50190114,HUDSON GARDEN GRILL,Bronx,American,2026-11-04T00:00:00.000,18,N
1,50188635,THE BORING SANDWICH,Manhattan,Sandwiches,2026-11-04T00:00:00.000,30,N
2,40692961,BAMBOO GARDEN RESTAURANT,Brooklyn,Fusion,2026-11-04T00:00:00.000,31,NaN
3,41454233,TASTY BITES,Brooklyn,Caribbean,2026-11-04T00:00:00.000,37,NaN
4,50105790,CHOMP CHOMP,Manhattan,Thai,2026-11-04T00:00:00.000,70,NaN


## The numeric column

`score` is the numeric column I will summarize. It is measured in inspection points, and lower values mean fewer violation points. I check its summary and confirm that there are no missing scores in the sample.

In [2]:
scores = inspections['score']
print('Rows in the sample:', len(inspections))
print('Missing scores:', scores.isna().sum())
scores.describe()

Rows in the sample: 1000
Missing scores: 0


count    1000.000000
mean       26.428000
std        15.972537
min         0.000000
25%        13.000000
50%        25.000000
75%        34.000000
max        75.000000
Name: score, dtype: float64

## Mean, median, and mode with pandas

Pandas gives quick answers. The mean is the arithmetic average, the median is the middle value after sorting, and the mode is the most frequent score.

In [3]:
score_mean = scores.mean()
score_median = scores.median()
score_mode = scores.mode().iloc[0]

print('Mean:', score_mean)
print('Median:', score_median)
print('Mode:', score_mode)

Mean: 26.428
Median: 25.0
Mode: 12


## Doing the same calculations the hard way

To make sure I understand the shortcuts, I repeat everything with only the Python standard library. This cell does not use pandas or the `statistics` module, and it does not call functions named `mean`, `median`, or `mode`. I read the CSV with `csv.DictReader` and use a dictionary to count how often each score appears.

In [4]:
import csv

raw_scores = []
with open('restaurant_inspections.csv', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['score'] != '':
            raw_scores.append(int(float(row['score'])))

total = 0
for value in raw_scores:
    total = total + value
manual_mean = total / len(raw_scores)

ordered = sorted(raw_scores)
middle = len(ordered) // 2
if len(ordered) % 2 == 1:
    manual_median = ordered[middle]
else:
    manual_median = (ordered[middle - 1] + ordered[middle]) / 2

counts = {}
for value in raw_scores:
    if value in counts:
        counts[value] = counts[value] + 1
    else:
        counts[value] = 1

highest_count = 0
manual_mode = None
for value in counts:
    if counts[value] > highest_count:
        highest_count = counts[value]
        manual_mode = value

print('Mean:', manual_mean)
print('Median:', manual_median)
print('Mode:', manual_mode)
print('Values read:', len(raw_scores))

Mean: 26.428
Median: 25.0
Mode: 12
Values read: 1000


## Checking that the two methods agree

The pandas and standard-library versions should produce the same three numbers. Putting them side by side also makes it easier to see that the mean is pulled upward by the high-score tail, while the median stays closer to the middle of the data.

In [5]:
comparison = pd.DataFrame({
    'method': ['pandas', 'standard library'],
    'mean': [score_mean, manual_mean],
    'median': [score_median, manual_median],
    'mode': [score_mode, manual_mode],
})
comparison

,method,mean,median,mode
0,pandas,26.428,25.0,12
1,standard library,26.428,25.0,12


## Visualizing the distribution

The chart below uses only the standard library. I divide the scores into 10-point bins, count how many inspections fall into each bin, and draw a horizontal bar using block characters. A longer bar means more inspections in that score range. This format stays readable on a narrow screen.

In [6]:
import math

bin_width = 10
highest_score = max(raw_scores)
number_of_bins = math.ceil((highest_score + 1) / bin_width)
bin_counts = [0] * number_of_bins

for value in raw_scores:
    index = value // bin_width
    bin_counts[index] = bin_counts[index] + 1

largest_bin = max(bin_counts)
chart_width = 40

print('Distribution of NYC restaurant inspection scores')
print()
print('Inspection score (points; lower is better)')
for index in range(number_of_bins):
    lower = index * bin_width
    upper = lower + bin_width - 1
    blocks = round(bin_counts[index] / largest_bin * chart_width)
    bar = '█' * blocks
    print(f'{lower:>3}-{upper:<3} | {bar} {bin_counts[index]}')
print()
print('Number of inspections (one number at the end of each row)')

Distribution of NYC restaurant inspection scores

Inspection score (points; lower is better)
  0-9   | ██████████████ 100
 10-19  | ██████████████████████████████████████ 282
 20-29  | ████████████████████████████████████████ 295
 30-39  | ██████████████████ 135
 40-49  | ██████████ 77
 50-59  | █████████ 67
 60-69  | ████ 28
 70-79  | ██ 16

Number of inspections (one number at the end of each row)


## What I take away

Most inspections in this sample fall in the lower score ranges, but there is a long tail of higher scores. The mean is higher than the median, which is a useful reminder that an average can be pulled up by a relatively small number of unusual observations.

A low score is good, but it is not proof that a restaurant is flawless. An inspection is a snapshot, and restaurants with repeated inspections can appear more often in the data. The sample also contains only the latest 1,000 inspections with a score, so it is not a random sample of every restaurant in the city. I dropped rows without a numeric score because there is nothing to average in those rows.

## Source and methods

- [NYC Restaurant Inspections dataset](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j) from the NYC Department of Health and Mental Hygiene.
- I used the Socrata API to select the latest 1,000 rows where `score` was not null, keeping `camis`, `dba`, `boro`, `cuisine_description`, `inspection_date`, `score`, and `grade`.
- The data dictionary is available as an attachment on the dataset page.